In [2]:
import os
import random
import shutil

dataset_path = "C:\Plant Classification Project\medicinalplant"           # your original dataset path
removed_path = "removed_images"     # new folder for removed images

os.makedirs(removed_path, exist_ok=True)

total_remove = 2500
classes = os.listdir(dataset_path)

remove_per_class = total_remove // len(classes)
extra_remove = total_remove % len(classes)

for i, class_name in enumerate(classes):
    class_path = os.path.join(dataset_path, class_name)
    removed_class_path = os.path.join(removed_path, class_name)

    if os.path.isdir(class_path):
        os.makedirs(removed_class_path, exist_ok=True)

        images = os.listdir(class_path)

        # Calculate how many to remove from this class
        num_to_remove = remove_per_class + (1 if i < extra_remove else 0)

        remove_images = random.sample(images, num_to_remove)

        for img in remove_images:
            src = os.path.join(class_path, img)
            dst = os.path.join(removed_class_path, img)
            shutil.move(src, dst)

print("✅ 2,500 images moved to 'removed_images' folder successfully!")


✅ 2,500 images moved to 'removed_images' folder successfully!


In [ ]:
import os
import random
import shutil

original_dataset = "C:\Plant Classification Project\medicinalplant"          # 20,000 images folder
output_base = "dataset_split"

train_path = os.path.join(output_base, "train")
val_path = os.path.join(output_base, "validation")
test_path = os.path.join(output_base, "test")

for path in [train_path, val_path, test_path]:
    os.makedirs(path, exist_ok=True)

for class_name in os.listdir(original_dataset):
    class_path = os.path.join(original_dataset, class_name)

    if os.path.isdir(class_path):
        images = os.listdir(class_path)
        random.shuffle(images)

        total = len(images)
        train_count = int(total * 0.70)
        val_count = int(total * 0.15)

        train_images = images[:train_count]
        val_images = images[train_count:train_count + val_count]
        test_images = images[train_count + val_count:]

        for folder in [train_path, val_path, test_path]:
            os.makedirs(os.path.join(folder, class_name), exist_ok=True)

        for img in train_images:
            shutil.copy(os.path.join(class_path, img),
                        os.path.join(train_path, class_name, img))

        for img in val_images:
            shutil.copy(os.path.join(class_path, img),
                        os.path.join(val_path, class_name, img))

        for img in test_images:
            shutil.copy(os.path.join(class_path, img),
                        os.path.join(test_path, class_name, img))

print("✅ Dataset split into 70–15–15 successfully!")


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model, load_model
import os


In [3]:


# 🔹 Training generator (with augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    "dataset_split/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

# 🔹 Validation generator (no augmentation)
val_datagen = ImageDataGenerator(rescale=1./255)

val_generator = val_datagen.flow_from_directory(
    "dataset_split/validation",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

# 🔹 Testing generator (only for final evaluation)
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    "dataset_split/test",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)



Found 16631 images belonging to 45 classes.
Found 4560 images belonging to 45 classes.
Found 4648 images belonging to 45 classes.


In [4]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Load pretrained MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 112, 112, 32)      │             864 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 112, 112, 32)      │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 112, 112, 32)      │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 32)      │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 112, 112, 32)      │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 112, 112, 32)      │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 112, 112, 96)      │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 112, 112, 96)      │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 112, 112, 96)      │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 113, 113, 96)      │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 56, 56, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,597,485 (9.91 MB)

 Trainable params: 339,501 (1.30 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [5]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=3
)

model.save("plant_model.keras")
print("Model saved after 3 epochs")



Epoch 1/3
520/520 ━━━━━━━━━━━━━━━━━━━━ 940s 2s/step - accuracy: 0.5549 - loss: 1.7660 - val_accuracy: 0.8899 - val_loss: 0.5664
Epoch 2/3
520/520 ━━━━━━━━━━━━━━━━━━━━ 2133s 4s/step - accuracy: 0.8319 - loss: 0.6196 - val_accuracy: 0.9329 - val_loss: 0.3096
Epoch 3/3
520/520 ━━━━━━━━━━━━━━━━━━━━ 2900s 6s/step - accuracy: 0.8877 - loss: 0.4010 - val_accuracy: 0.9537 - val_loss: 0.2105
Model saved after 3 epochs
